# RW-Causal Forest Pipeline




## 1. Setup

In [ ]:
!pip install dowhy econml scikit-learn pandas numpy scipy shap openpyxl matplotlib -q

In [ ]:
import re
import time
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.metrics import (average_precision_score, f1_score, recall_score,
                              confusion_matrix, ConfusionMatrixDisplay,
                              precision_recall_curve, roc_curve, auc)
from scipy.stats import wilcoxon
from econml.dml import CausalForestDML
import shap

np.random.seed(42)


In [ ]:
!pip freeze | grep -iE "^dowhy|^econml|^scikit-learn|^pandas|^numpy|^scipy|^shap|^openpyxl|^matplotlib"
import sys
print("Python version:", sys.version)


## 2. Helper Functions

In [ ]:
# 2a. Prepare one fold with no leakage: scaler and class weight fit on the
# training portion only. Optionally applies an in-fold overlap trim (School
# Type), fitting the propensity model on the training partition only and
# applying it to both sides. Groups are carried through and re-aligned after
# trimming so group-aware splitting downstream stays valid.

def prepare_fold(X_raw, T, Y, G, train_idx, test_idx, needs_trim=False,
                  trim_lo=0.05, trim_hi=0.95):
    X_train_raw, X_test_raw = X_raw[train_idx], X_raw[test_idx]
    T_train, T_test = T[train_idx], T[test_idx]
    Y_train, Y_test = Y[train_idx], Y[test_idx]
    G_train, G_test = G[train_idx], G[test_idx]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_test = scaler.transform(X_test_raw)

    if needs_trim:
        overlap_model = LogisticRegression(max_iter=1000)
        overlap_model.fit(X_train, T_train)
        p_train = overlap_model.predict_proba(X_train)[:, 1]
        p_test = overlap_model.predict_proba(X_test)[:, 1]
        keep_train = (p_train >= trim_lo) & (p_train <= trim_hi)
        keep_test = (p_test >= trim_lo) & (p_test <= trim_hi)
        print(f"    in-fold overlap trim | train {len(X_train)}->{keep_train.sum()} "
              f"| test {len(X_test)}->{keep_test.sum()}")
        X_train, T_train, Y_train, G_train = (X_train[keep_train], T_train[keep_train],
                                               Y_train[keep_train], G_train[keep_train])
        X_test, T_test, Y_test, G_test = (X_test[keep_test], T_test[keep_test],
                                           Y_test[keep_test], G_test[keep_test])

    n1 = (Y_train == 1).sum()
    n0 = (Y_train == 0).sum()
    ratio = max(n1, n0) / min(n1, n0) if min(n1, n0) > 0 else 1.0
    quarter_weight = {0: 1, 1: 1 + (ratio - 1) * 0.25}

    return X_train, X_test, T_train, T_test, Y_train, Y_test, G_train, G_test, quarter_weight


In [ ]:
# 2b. Fit one CausalForestDML model. LEAF_ENGINEERED = 12 per the supervisor
# ruling (validated against the full factorial search average of 12.8%
# CI-width reduction, while retaining more CATE heterogeneity than leaf=15/20).
LEAF_BASELINE = 5
LEAF_ENGINEERED = 12
WEIGHT_FRAC = 0.25

def fit_causal_forest(X, T, Y, min_samples_leaf, class_weight=None):
    model = CausalForestDML(
        model_y=LogisticRegression(class_weight=class_weight, max_iter=1000),
        model_t=RandomForestClassifier(random_state=42),
        discrete_outcome=True,
        discrete_treatment=True,
        n_estimators=500,
        min_samples_leaf=min_samples_leaf,
        max_depth=8,
        honest=True,
        random_state=42
    )
    model.fit(Y, T, X=X)
    return model


In [ ]:
# 2c. Compute all six metrics for one fitted model
METRIC_KEYS = ["CI width", "AUC-PR", "Macro F1", "At-risk recall",
               "Equal Opportunity diff", "Equalized Odds diff"]

def evaluate_model(model, X_test, Y_test, T_test, T0=0, T1=1):
    lower, upper = model.effect_interval(X_test, T0=T0, T1=T1, alpha=0.05)
    ci_width = (upper - lower).mean()

    probs = model.models_y[0][0].predict_proba(X_test)[:, 1]
    preds = (probs >= 0.5).astype(int)

    auc_pr = average_precision_score(Y_test, probs)
    macro_f1 = f1_score(Y_test, preds, average="macro")
    recall = recall_score(Y_test, preds, pos_label=1)

    group0 = (T_test == T0)
    group1 = (T_test == T1)

    tpr0 = preds[group0][Y_test[group0] == 1].mean() if (Y_test[group0] == 1).sum() > 0 else np.nan
    tpr1 = preds[group1][Y_test[group1] == 1].mean() if (Y_test[group1] == 1).sum() > 0 else np.nan
    equal_opportunity_diff = tpr1 - tpr0

    fpr0 = preds[group0][Y_test[group0] == 0].mean() if (Y_test[group0] == 0).sum() > 0 else np.nan
    fpr1 = preds[group1][Y_test[group1] == 0].mean() if (Y_test[group1] == 0).sum() > 0 else np.nan
    equalized_odds_diff = max(abs(tpr1 - tpr0), abs(fpr1 - fpr0))

    return {
        "CI width": ci_width, "AUC-PR": auc_pr, "Macro F1": macro_f1,
        "At-risk recall": recall, "Equal Opportunity diff": equal_opportunity_diff,
        "Equalized Odds diff": equalized_odds_diff,
    }


In [ ]:
# 2d. Plotting functions
BASELINE_COLOR = "#c44e52"
ENGINEERED_COLOR = "#4c72b0"

def plot_confusion_matrices(Y_true, base_preds, eng_preds, label):
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    ConfusionMatrixDisplay.from_predictions(Y_true, base_preds, normalize="true",
                                             ax=axes[0], colorbar=False, cmap="Reds")
    axes[0].set_title(f"{label} - Baseline")
    ConfusionMatrixDisplay.from_predictions(Y_true, eng_preds, normalize="true",
                                             ax=axes[1], colorbar=False, cmap="Blues")
    axes[1].set_title(f"{label} - Engineered")
    plt.tight_layout()
    safe_name = label.replace(" ", "_").replace("-", "").replace("/", "_")
    plt.savefig(f"confusion_matrix_{safe_name}.png", dpi=300)
    plt.show()

def plot_pr_roc_curves(Y_true, base_probs, eng_probs, label):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    base_p, base_r, _ = precision_recall_curve(Y_true, base_probs)
    eng_p, eng_r, _ = precision_recall_curve(Y_true, eng_probs)
    base_ap = average_precision_score(Y_true, base_probs)
    eng_ap = average_precision_score(Y_true, eng_probs)
    axes[0].plot(base_r, base_p, color=BASELINE_COLOR, label=f"Baseline (AUC-PR={base_ap:.3f})")
    axes[0].plot(eng_r, eng_p, color=ENGINEERED_COLOR, label=f"Engineered (AUC-PR={eng_ap:.3f})")
    axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision")
    axes[0].set_title(f"{label} - Precision-Recall")
    axes[0].legend()

    base_fpr, base_tpr, _ = roc_curve(Y_true, base_probs)
    eng_fpr, eng_tpr, _ = roc_curve(Y_true, eng_probs)
    base_auc = auc(base_fpr, base_tpr)
    eng_auc = auc(eng_fpr, eng_tpr)
    axes[1].plot(base_fpr, base_tpr, color=BASELINE_COLOR, label=f"Baseline (AUC={base_auc:.3f})")
    axes[1].plot(eng_fpr, eng_tpr, color=ENGINEERED_COLOR, label=f"Engineered (AUC={eng_auc:.3f})")
    axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=0.8)
    axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
    axes[1].set_title(f"{label} - ROC")
    axes[1].legend()

    plt.tight_layout()
    safe_name = label.replace(" ", "_").replace("-", "").replace("/", "_")
    plt.savefig(f"pr_roc_curves_{safe_name}.png", dpi=300)
    plt.show()

def plot_fold_level_boxplot(baseline_values, engineered_values, label):
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    data = [baseline_values, engineered_values]
    bp = ax.boxplot(data, tick_labels=["Baseline", "Engineered"],
                     showmeans=True, meanline=True, patch_artist=True)
    for patch, color in zip(bp["boxes"], [BASELINE_COLOR, ENGINEERED_COLOR]):
        patch.set_facecolor(color)
        patch.set_alpha(0.5)
    for i, vals in enumerate(data, start=1):
        x = np.random.normal(i, 0.04, size=len(vals))
        ax.scatter(x, vals, alpha=0.6, s=14, color="black", zorder=3)
    ax.set_ylabel("CATE CI width")
    ax.set_title(f"{label}\n({len(baseline_values)} points per model)")
    plt.tight_layout()
    safe_name = label.replace(" ", "_").replace("-", "").replace("/", "_")
    plt.savefig(f"fold_level_boxplot_{safe_name}.png", dpi=300)
    plt.show()


In [ ]:
# 2e. Full evaluation: GROUPED search/confirm split, GROUPED repeated CV,
# optional in-fold trim, native plotting. This is the corrected replacement
# for the earlier ungrounded (student-level) splitting.

def run_full_evaluation(X_raw, T, Y, G, label, comparisons=((0, 1),), n_repeats=10,
                         needs_trim=False):
    # Grouped, approximately-stratified 80/20 search/confirm split
    outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    search_idx, confirm_idx = next(outer.split(np.zeros(len(Y)), Y, groups=G))

    X_search_raw, T_search, Y_search, G_search = X_raw[search_idx], T[search_idx], Y[search_idx], G[search_idx]

    results = {c: {"baseline": {k: [] for k in METRIC_KEYS},
                    "engineered": {k: [] for k in METRIC_KEYS}} for c in comparisons}
    train_times = {"baseline": [], "engineered": []}

    for repeat in range(n_repeats):
        skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=repeat)
        for tr_idx, te_idx in skf.split(np.zeros(len(Y_search)), Y_search, groups=G_search):
            X_tr, X_te, T_tr, T_te, Y_tr, Y_te, G_tr, G_te, qw = prepare_fold(
                X_search_raw, T_search, Y_search, G_search, tr_idx, te_idx, needs_trim=needs_trim
            )
            if len(Y_tr) == 0 or len(Y_te) == 0 or len(set(T_tr)) < 2:
                continue

            t0 = time.time()
            baseline_model = fit_causal_forest(X_tr, T_tr, Y_tr, min_samples_leaf=LEAF_BASELINE)
            train_times["baseline"].append(time.time() - t0)

            t0 = time.time()
            engineered_model = fit_causal_forest(X_tr, T_tr, Y_tr, min_samples_leaf=LEAF_ENGINEERED, class_weight=qw)
            train_times["engineered"].append(time.time() - t0)

            for comp in comparisons:
                T0, T1 = comp
                if (T_te == T0).sum() == 0 or (T_te == T1).sum() == 0:
                    continue
                b_scores = evaluate_model(baseline_model, X_te, Y_te, T_te, T0, T1)
                e_scores = evaluate_model(engineered_model, X_te, Y_te, T_te, T0, T1)
                for k in METRIC_KEYS:
                    results[comp]["baseline"][k].append(b_scores[k])
                    results[comp]["engineered"][k].append(e_scores[k])

    print(f"===== {label}: repeated grouped cross-validation ({n_repeats} repeats x 5 folds) =====")
    for comp in comparisons:
        print(f"--- treatment {comp[1]} vs {comp[0]} ---")
        table = {}
        for k in METRIC_KEYS:
            b_vals = np.array(results[comp]["baseline"][k])
            e_vals = np.array(results[comp]["engineered"][k])
            table[k] = [f"{np.nanmean(b_vals):.4f} +/- {np.nanstd(b_vals):.4f} (n={len(b_vals)})",
                        f"{np.nanmean(e_vals):.4f} +/- {np.nanstd(e_vals):.4f} (n={len(e_vals)})"]
        print(pd.DataFrame(table, index=["Baseline", "Engineered"]).T)
        print()

    print(f"Mean training time per fit: baseline {np.mean(train_times['baseline']):.3f}s "
          f"+/- {np.std(train_times['baseline']):.3f}s | "
          f"engineered {np.mean(train_times['engineered']):.3f}s "
          f"+/- {np.std(train_times['engineered']):.3f}s")
    print()

    plot_fold_level_boxplot(
        results[comparisons[0]]["baseline"]["CI width"],
        results[comparisons[0]]["engineered"]["CI width"],
        label=label
    )

    b_ci = np.array(results[comparisons[0]]["baseline"]["CI width"])
    e_ci = np.array(results[comparisons[0]]["engineered"]["CI width"])
    stat, p_value = wilcoxon(b_ci, e_ci)
    diffs = e_ci - b_ci
    pooled_std = np.sqrt((b_ci.std() ** 2 + e_ci.std() ** 2) / 2)
    cohens_d_unpaired = diffs.mean() / pooled_std if pooled_std > 0 else float("nan")
    cohens_d_paired = diffs.mean() / diffs.std(ddof=1) if diffs.std(ddof=1) > 0 else float("nan")

    boot_means = [np.random.choice(diffs, size=len(diffs), replace=True).mean() for _ in range(1000)]
    ci_lower, ci_upper = np.percentile(boot_means, 2.5), np.percentile(boot_means, 97.5)

    print(f"Wilcoxon signed-rank test on CI width: statistic={stat:.4f}, p={p_value:.6e}")
    print(f"Cohen's d_z (paired, matches the design): {cohens_d_paired:.4f}")
    print(f"Cohen's d (unpaired, for reference only): {cohens_d_unpaired:.4f}")
    print(f"95% bootstrap CI on mean CI-width difference: [{ci_lower:.4f}, {ci_upper:.4f}]")
    print()

    # Held-out confirmation: fit once on the full (grouped) search partition
    X_tr, X_te, T_tr, T_te, Y_tr, Y_te, G_tr, G_te, qw_confirm = prepare_fold(
        X_raw, T, Y, G, search_idx, confirm_idx, needs_trim=needs_trim
    )
    confirm_baseline = fit_causal_forest(X_tr, T_tr, Y_tr, min_samples_leaf=LEAF_BASELINE)
    confirm_engineered = fit_causal_forest(X_tr, T_tr, Y_tr, min_samples_leaf=LEAF_ENGINEERED, class_weight=qw_confirm)

    print(f"===== {label}: held-out confirmation (grouped, data never used in search) =====")
    for comp in comparisons:
        T0, T1 = comp
        b_conf = evaluate_model(confirm_baseline, X_te, Y_te, T_te, T0, T1)
        e_conf = evaluate_model(confirm_engineered, X_te, Y_te, T_te, T0, T1)
        print(f"--- treatment {T1} vs {T0} ---")
        print(pd.DataFrame({"Baseline": b_conf, "Engineered": e_conf}))
        print()

    b_probs = confirm_baseline.models_y[0][0].predict_proba(X_te)[:, 1]
    b_preds = (b_probs >= 0.5).astype(int)
    e_probs = confirm_engineered.models_y[0][0].predict_proba(X_te)[:, 1]
    e_preds = (e_probs >= 0.5).astype(int)

    print("Baseline confusion matrix (normalized by true class):")
    print(confusion_matrix(Y_te, b_preds, normalize="true"))
    print("Engineered confusion matrix (normalized by true class):")
    print(confusion_matrix(Y_te, e_preds, normalize="true"))

    plot_confusion_matrices(Y_te, b_preds, e_preds, label=label)
    plot_pr_roc_curves(Y_te, b_probs, e_probs, label=label)

    return confirm_baseline, confirm_engineered, X_te, Y_te, T_te


## 3. GhEduData Data

In [ ]:
ghedudata = pd.read_excel("GhEduData_Merged_Anonymized.xlsx", sheet_name="GhEduData_Merged")
print("Students loaded:", len(ghedudata))
print("Schools:", ghedudata["School_ID"].nunique())


In [ ]:
def clean_text(value):
    value = str(value).replace("\u25a1", "").strip()
    return " ".join(value.split())

messy_columns = ["Governance_Type", "Location_Category", "Avg_Class_Size", "Pct_Teachers_Degree",
                  "Teacher_Student_Ratio", "Classroom_Condition", "English_Textbook_Access",
                  "Math_Textbook_Access", "Home_Study_Access", "Socioeconomic_Profile"]

for col in messy_columns:
    ghedudata[col] = ghedudata[col].apply(clean_text)


In [ ]:
ghedudata["at_risk"] = (ghedudata["Aggregate"] >= 21).astype(int)
ghedudata["gender_num"] = ghedudata["Gender"].apply(lambda x: 1 if x == "Male" else 0)

def encode_location(value):
    levels = {"Rural": 0, "Peri-urban": 1, "Urban": 2}
    return levels.get(value, -1)

ghedudata["location_num"] = ghedudata["Location_Category"].apply(encode_location)
ghedudata["school_type_num"] = ghedudata["Governance_Type"].apply(lambda x: 1 if "Private" in x else 0)

print("At-risk count:", ghedudata["at_risk"].sum(), "out of", len(ghedudata))


In [ ]:
def encode_attendance(v):
    if "Less than 50%" in v: return 0
    if "50" in v and "74" in v: return 1
    if "75" in v and "90" in v: return 2
    if "More than 90%" in v: return 3
    return -1

def encode_homework(v):
    if "Rarely" in v: return 0
    if "Sometimes" in v: return 1
    if "Often" in v: return 2
    if "Almost always" in v: return 3
    return -1

def encode_participation(v):
    if "Passive" in v: return 0
    if "Very active" in v: return 3
    if "Moderate" in v: return 1
    if "Active" in v: return 2
    return -1

ghedudata["attendance_num"] = ghedudata["Attendance Rate"].apply(encode_attendance)
ghedudata["homework_num"] = ghedudata["Homework Submission Rate"].apply(encode_homework)
ghedudata["participation_num"] = ghedudata["Classroom Participation"].apply(encode_participation)

student_level_confounders = ["Age", "attendance_num", "homework_num", "participation_num"]


In [ ]:
def encode_class_size(v):
    sizes = {"Fewer than 20": 0, "20\u201335": 1, "36\u201350": 2,
             "51 - 65": 3, "66 - 80": 4, "More than 80": 5}
    return sizes.get(v, -1)

def encode_teacher_qual(v):
    quals = {"Less than 25%": 0, "50\u201375%": 1, "More than 75%": 2}
    return quals.get(v, -1)

def encode_teacher_ratio(v):
    if "fewer than 25" in v: return 0
    if "25\u201335" in v: return 1
    if "36\u201350" in v: return 2
    if "51 - 65" in v: return 3
    if "66 - 80" in v: return 4
    return -1

def encode_condition(v):
    if "Some classrooms are inadequate" in v: return 0
    if "Mostly adequate" in v: return 1
    if "All classrooms adequate" in v: return 2
    return -1

def encode_textbook(v):
    if "3 or more" in v: return 0
    if "between 2" in v: return 1
    if "own copy" in v: return 2
    return -1

def encode_home_study(v):
    if "Fewer than 25%" in v: return 0
    if "25\u201349%" in v: return 1
    if "More than 75%" in v: return 3
    if "50" in v: return 2
    return -1

def encode_socioeconomic(v):
    if "subsistence" in v: return 0
    if "middle income" in v: return 2
    if "high income" in v: return 3
    if "low income" in v: return 1
    return -1

ghedudata["class_size_num"] = ghedudata["Avg_Class_Size"].apply(encode_class_size)
ghedudata["teacher_qual_num"] = ghedudata["Pct_Teachers_Degree"].apply(encode_teacher_qual)
ghedudata["teacher_ratio_num"] = ghedudata["Teacher_Student_Ratio"].apply(encode_teacher_ratio)
ghedudata["classroom_condition_num"] = ghedudata["Classroom_Condition"].apply(encode_condition)
ghedudata["english_textbook_num"] = ghedudata["English_Textbook_Access"].apply(encode_textbook)
ghedudata["math_textbook_num"] = ghedudata["Math_Textbook_Access"].apply(encode_textbook)
ghedudata["home_study_num"] = ghedudata["Home_Study_Access"].apply(encode_home_study)
ghedudata["socioeconomic_num"] = ghedudata["Socioeconomic_Profile"].apply(encode_socioeconomic)

school_level_confounders = ["class_size_num", "teacher_qual_num", "teacher_ratio_num",
                             "classroom_condition_num", "english_textbook_num",
                             "math_textbook_num", "home_study_num", "socioeconomic_num"]

full_confounders = student_level_confounders + school_level_confounders

for col in ["attendance_num", "homework_num", "participation_num"] + school_level_confounders:
    unmatched = (ghedudata[col] == -1).sum()
    if unmatched > 0:
        print("WARNING - unmatched values in", col, ":", unmatched)
print("Encoding check complete.")


In [ ]:
# No school-type overlap trim here - it is now computed IN-FOLD inside
# run_full_evaluation (needs_trim=True), not upfront on the full corpus.

X_gh_full_raw = ghedudata[full_confounders].values
X_gh_student_raw = ghedudata[student_level_confounders].values

Y_gh = ghedudata["at_risk"].values
T_gh_gender = ghedudata["gender_num"].values
T_gh_location = ghedudata["location_num"].values
T_gh_schooltype = ghedudata["school_type_num"].values
G_gh = ghedudata["School_ID"].values

print("GhEduData ready:", X_gh_full_raw.shape, "| schools:", len(set(G_gh)))


## 4. GhEduData Results

In [ ]:
# 4a. Gender: grouped repeated CV + grouped held-out confirmation
gh_gender_baseline, gh_gender_engineered, X_gh_gender_confirm, Y_gh_gender_confirm, T_gh_gender_confirm = run_full_evaluation(
    X_gh_full_raw, T_gh_gender, Y_gh, G_gh, label="GhEduData - Gender", n_repeats=10
)


In [ ]:
# 4b. Location: grouped repeated CV + grouped held-out confirmation
gh_loc_baseline, gh_loc_engineered, X_gh_loc_confirm, Y_gh_loc_confirm, T_gh_loc_confirm = run_full_evaluation(
    X_gh_student_raw, T_gh_location, Y_gh, G_gh, label="GhEduData - Location",
    comparisons=((0, 1), (0, 2)), n_repeats=10
)


In [ ]:
# 4c. School type: grouped repeated CV, IN-FOLD overlap trim, grouped confirmation
gh_school_baseline, gh_school_engineered, X_gh_school_confirm, Y_gh_school_confirm, T_gh_school_confirm = run_full_evaluation(
    X_gh_student_raw, T_gh_schooltype, Y_gh, G_gh,
    label="GhEduData - School Type", n_repeats=10, needs_trim=True
)


## 5. OULAD Data




In [ ]:
url = "https://archive.ics.uci.edu/static/public/349/open+university+learning+analytics+dataset.zip"
urllib.request.urlretrieve(url, "oulad.zip")
with zipfile.ZipFile("oulad.zip", "r") as zip_ref:
    zip_ref.extractall("oulad_data")

student_info = pd.read_csv("oulad_data/studentInfo.csv")
student_vle = pd.read_csv("oulad_data/studentVle.csv")
student_assessment = pd.read_csv("oulad_data/studentAssessment.csv")

print("Rows loaded:", len(student_info), "| distinct students:", student_info["id_student"].nunique())


In [ ]:
CUTOFF_DAY = 28   # chosen from the retention/leakage table (RUN 06b Part 1), before any CATE was inspected

student_info["at_risk"] = student_info["final_result"].apply(
    lambda x: 0 if x in ["Pass", "Distinction"] else 1
)
student_info["gender_num"] = student_info["gender"].apply(lambda x: 1 if x == "M" else 0)

# Regime B: DROP records with missing treatment, do not impute
n_before = len(student_info)
student_info = student_info[student_info["imd_band"].astype(str) != "?"].copy()
print(f"FILTER drop missing treatment (imd_band) | n before {n_before} | n after {len(student_info)}")

def imd_lower_bound(band_text):
    match = re.search(r"\d+", str(band_text))
    return int(match.group())

student_info["imd_lower_bound"] = student_info["imd_band"].apply(imd_lower_bound)

def group_deprivation(lower_bound):
    if lower_bound <= 20:
        return 0
    elif lower_bound <= 60:
        return 1
    else:
        return 2

student_info["location_num"] = student_info["imd_lower_bound"].apply(group_deprivation)


In [ ]:
def encode_education(level):
    levels = {"No Formal quals": 0, "Lower Than A Level": 1, "A Level or Equivalent": 2,
              "HE Qualification": 3, "Post Graduate Qualification": 4}
    return levels.get(level, -1)

student_info["education_num"] = student_info["highest_education"].apply(encode_education)

def encode_age(band):
    bands = {"0-35": 0, "35-55": 1, "55<=": 2}
    return bands.get(band, -1)

student_info["age_num"] = student_info["age_band"].apply(encode_age)

# Regime B: engagement window applied BEFORE aggregation, merge on the FULL
# presentation key (not id_student alone)
vle_windowed = student_vle[student_vle["date"] <= CUTOFF_DAY]
print(f"FILTER engagement window date<={CUTOFF_DAY} | n before {len(student_vle)} | n after {len(vle_windowed)}")

assess_windowed = student_assessment[student_assessment["date_submitted"] <= CUTOFF_DAY]
print(f"FILTER submissions date_submitted<={CUTOFF_DAY} | n before {len(student_assessment)} | n after {len(assess_windowed)}")

engagement = vle_windowed.groupby(["id_student", "code_module", "code_presentation"]).agg(
    total_clicks=("sum_click", "sum"),
    active_days=("date", "nunique")
).reset_index()

submitted = assess_windowed.groupby("id_student").size().reset_index(name="submitted_count")

oulad_data = student_info.merge(engagement, on=["id_student", "code_module", "code_presentation"], how="left")
print(f"MERGE engagement (full presentation key) | n before {len(student_info)} | n after {len(oulad_data)}")
oulad_data = oulad_data.merge(submitted, on="id_student", how="left")
print(f"MERGE submissions | n before {len(student_info)} | n after {len(oulad_data)}")

oulad_data["total_clicks"] = oulad_data["total_clicks"].fillna(0)
oulad_data["active_days"] = oulad_data["active_days"].fillna(0)
oulad_data["submitted_count"] = oulad_data["submitted_count"].fillna(0)

print("Any unmatched education or age values?",
      (oulad_data["education_num"] == -1).sum(), (oulad_data["age_num"] == -1).sum())


In [ ]:
oulad_confounders = ["education_num", "age_num", "total_clicks", "active_days", "submitted_count"]

X_oulad_raw = oulad_data[oulad_confounders].values
T_oulad_gender = oulad_data["gender_num"].values
T_oulad_location = oulad_data["location_num"].values
Y_oulad = oulad_data["at_risk"].values
G_oulad = oulad_data["id_student"].values   # grouped by STUDENT, not by row

print("OULAD ready:", X_oulad_raw.shape, "| at-risk rate:", Y_oulad.mean().round(3),
      "| distinct students:", len(set(G_oulad)))


## 6. OULAD Results 

In [ ]:
# 6a. Gender: grouped repeated CV + grouped held-out confirmation
oulad_gender_baseline, oulad_gender_engineered, X_oulad_gender_confirm, Y_oulad_gender_confirm, T_oulad_gender_confirm = run_full_evaluation(
    X_oulad_raw, T_oulad_gender, Y_oulad, G_oulad, label="OULAD - Gender", n_repeats=5
)


In [ ]:
# 6b. Location: grouped repeated CV + grouped held-out confirmation
oulad_loc_baseline, oulad_loc_engineered, X_oloc_confirm, Y_oloc_confirm, T_oloc_confirm = run_full_evaluation(
    X_oulad_raw, T_oulad_location, Y_oulad, G_oulad, label="OULAD - Location",
    comparisons=((0, 1), (0, 2)), n_repeats=5
)


## 7. Ablation Isolation

In [ ]:
# 7a. Isolate each engineered component separately, GhEduData Gender,
# using the grouped, corrected splitting protocol throughout.
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
search_idx, confirm_idx = next(outer.split(np.zeros(len(Y_gh)), Y_gh, groups=G_gh))
X_search_raw = X_gh_full_raw[search_idx]
T_search = T_gh_gender[search_idx]
Y_search = Y_gh[search_idx]
G_search = G_gh[search_idx]

ablation_results = {c: {k: [] for k in METRIC_KEYS} for c in ["baseline", "leaf_only", "weight_only", "full"]}

for repeat in range(10):
    skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=repeat)
    for tr_idx, te_idx in skf.split(np.zeros(len(Y_search)), Y_search, groups=G_search):
        X_tr, X_te, T_tr, T_te, Y_tr, Y_te, G_tr, G_te, qw = prepare_fold(
            X_search_raw, T_search, Y_search, G_search, tr_idx, te_idx
        )
        if len(Y_tr) == 0 or len(Y_te) == 0 or len(set(T_tr)) < 2:
            continue

        m_baseline = fit_causal_forest(X_tr, T_tr, Y_tr, min_samples_leaf=LEAF_BASELINE)
        m_leaf_only = fit_causal_forest(X_tr, T_tr, Y_tr, min_samples_leaf=LEAF_ENGINEERED)
        m_weight_only = fit_causal_forest(X_tr, T_tr, Y_tr, min_samples_leaf=LEAF_BASELINE, class_weight=qw)
        m_full = fit_causal_forest(X_tr, T_tr, Y_tr, min_samples_leaf=LEAF_ENGINEERED, class_weight=qw)

        for name, model in [("baseline", m_baseline), ("leaf_only", m_leaf_only),
                             ("weight_only", m_weight_only), ("full", m_full)]:
            scores = evaluate_model(model, X_te, Y_te, T_te)
            for k in METRIC_KEYS:
                ablation_results[name][k].append(scores[k])

print("=== Ablation isolation: GhEduData - Gender (grouped, 10 repeats x 5 folds) ===")
table = {}
for name in ["baseline", "leaf_only", "weight_only", "full"]:
    table[name] = [f"{np.nanmean(ablation_results[name][k]):.4f} +/- {np.nanstd(ablation_results[name][k]):.4f}"
                   for k in METRIC_KEYS]
ablation_table = pd.DataFrame(table, index=METRIC_KEYS)
print(ablation_table)


In [ ]:
# 7b. Plot the ablation comparison for the primary metric (CI width)
fig, ax = plt.subplots(figsize=(6, 4.5))
conditions = ["baseline", "leaf_only", "weight_only", "full"]
means = [np.nanmean(ablation_results[c]["CI width"]) for c in conditions]
sds = [np.nanstd(ablation_results[c]["CI width"]) for c in conditions]
colors = ["#c44e52", "#dd8452", "#8172b3", "#4c72b0"]

ax.bar(conditions, means, yerr=sds, capsize=5, color=colors, alpha=0.8)
ax.set_ylabel("CATE CI width (mean +/- SD)")
ax.set_title("GhEduData - Gender: ablation isolation (leaf=12)")
plt.tight_layout()
plt.savefig("ablation_ci_width_barplot.png", dpi=300)
plt.show()


## 8. SHAP Equity Diagnostics

In [ ]:
def run_shap_diagnostics(model, X_confirm, feature_names, label):
    shap_dict = model.shap_values(X_confirm, feature_names=feature_names)
    outcome_key = list(shap_dict.keys())[0]
    treatment_key = list(shap_dict[outcome_key].keys())[0]
    explanation = shap_dict[outcome_key][treatment_key]

    mean_abs_shap = np.abs(explanation.values).mean(axis=0)
    ranking = sorted(zip(explanation.feature_names, mean_abs_shap), key=lambda x: -x[1])

    print(f"=== SHAP ranking: {label} ===")
    for name, value in ranking:
        print(f"{name}: {value:.4f}")
    print()

    fig, ax = plt.subplots(figsize=(6, max(2.5, 0.35 * len(ranking))))
    names = [r[0] for r in ranking][::-1]
    values = [r[1] for r in ranking][::-1]
    ax.barh(names, values, color="#4c72b0")
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_title(label)
    plt.tight_layout()
    safe_name = label.replace(" ", "_").replace("-", "").replace("/", "_")
    plt.savefig(f"shap_ranking_{safe_name}.png", dpi=300)
    plt.show()

    return explanation


In [ ]:
gh_gender_shap = run_shap_diagnostics(
    gh_gender_engineered, X_gh_gender_confirm, full_confounders, label="GhEduData - Gender"
)


In [ ]:
gh_loc_shap = run_shap_diagnostics(
    gh_loc_engineered, X_gh_loc_confirm, student_level_confounders, label="GhEduData - Location"
)


In [ ]:
gh_school_shap = run_shap_diagnostics(
    gh_school_engineered, X_gh_school_confirm, student_level_confounders, label="GhEduData - School Type"
)


In [ ]:
oulad_gender_shap = run_shap_diagnostics(
    oulad_gender_engineered, X_oulad_gender_confirm, oulad_confounders, label="OULAD - Gender"
)


In [ ]:
oulad_loc_shap = run_shap_diagnostics(
    oulad_loc_engineered, X_oloc_confirm, oulad_confounders, label="OULAD - Location"
)


## 9. Average Treatment Effects 




In [ ]:
def report_ate(model, X_test, T0, T1, label):
    ate = model.ate(X_test, T0=T0, T1=T1)
    lo, hi = model.ate_interval(X_test, T0=T0, T1=T1, alpha=0.05)
    ate_val = float(np.asarray(ate).ravel()[0]) if np.ndim(ate) > 0 else float(ate)
    lo_val = float(np.asarray(lo).ravel()[0]) if np.ndim(lo) > 0 else float(lo)
    hi_val = float(np.asarray(hi).ravel()[0]) if np.ndim(hi) > 0 else float(hi)
    crosses_zero = lo_val <= 0 <= hi_val
    verdict = ("CANNOT rule out zero effect - INCONCLUSIVE" if crosses_zero
               else f"excludes zero - {'INCREASES' if ate_val > 0 else 'DECREASES'} at-risk probability")
    print(f"  {label:<32} ATE = {ate_val:+.4f} | 95% CI [{lo_val:.4f}, {hi_val:.4f}] | {verdict}")
    return dict(label=label, ate=ate_val, ci_lo=lo_val, ci_hi=hi_val, crosses_zero=crosses_zero)

ate_rows = []

print("=== GhEduData: Average Treatment Effects (held-out confirmation partition) ===")
ate_rows.append(report_ate(gh_gender_baseline, X_gh_gender_confirm, 0, 1, "Gender (baseline) - Male vs Female"))
ate_rows.append(report_ate(gh_gender_engineered, X_gh_gender_confirm, 0, 1, "Gender (engineered) - Male vs Female"))
ate_rows.append(report_ate(gh_loc_baseline, X_gh_loc_confirm, 0, 1, "Location (baseline) - Peri-urban vs Rural"))
ate_rows.append(report_ate(gh_loc_engineered, X_gh_loc_confirm, 0, 1, "Location (engineered) - Peri-urban vs Rural"))
ate_rows.append(report_ate(gh_loc_baseline, X_gh_loc_confirm, 0, 2, "Location (baseline) - Urban vs Rural"))
ate_rows.append(report_ate(gh_loc_engineered, X_gh_loc_confirm, 0, 2, "Location (engineered) - Urban vs Rural"))
ate_rows.append(report_ate(gh_school_baseline, X_gh_school_confirm, 0, 1, "School Type (baseline) - Private vs Public"))
ate_rows.append(report_ate(gh_school_engineered, X_gh_school_confirm, 0, 1, "School Type (engineered) - Private vs Public"))

print()
print("=== OULAD: Average Treatment Effects (held-out confirmation partition) ===")
ate_rows.append(report_ate(oulad_gender_baseline, X_oulad_gender_confirm, 0, 1, "Gender (baseline) - Male vs Female"))
ate_rows.append(report_ate(oulad_gender_engineered, X_oulad_gender_confirm, 0, 1, "Gender (engineered) - Male vs Female"))
ate_rows.append(report_ate(oulad_loc_baseline, X_oloc_confirm, 0, 1, "Location (baseline) - Mid vs Low deprivation"))
ate_rows.append(report_ate(oulad_loc_engineered, X_oloc_confirm, 0, 1, "Location (engineered) - Mid vs Low deprivation"))
ate_rows.append(report_ate(oulad_loc_baseline, X_oloc_confirm, 0, 2, "Location (baseline) - High vs Low deprivation"))
ate_rows.append(report_ate(oulad_loc_engineered, X_oloc_confirm, 0, 2, "Location (engineered) - High vs Low deprivation"))

ate_df = pd.DataFrame(ate_rows)
ate_df.to_csv("DORAMP_ate_results.csv", index=False)
print()
print("WROTE DORAMP_ate_results.csv")
print()
print("A treatment's causal claim is only supportable where 'crosses_zero' is False.")
print("Where it is True, the honest wording is 'inconclusive', never 'no effect'.")
